# Programme 2 — Auslöschung (cancellation catastrophique)

**Référence :** Kröger, *Numerische Mathematik 1 für AMP*, sections 1.3.3 et 1.4.

Ce notebook accompagne le module `cancellation.py` et illustre :

1. La condition de l'addition (formule 1.2 du script)
2. Le **Beispiel 1.13** du script : $\sqrt{x^2+1} - x$ instable vs reformulation stable
3. Un autre cas classique : $(1 - \cos x)/x^2$ près de zéro
4. Le compromis troncature/Auslöschung dans la dérivée numérique
5. La variance "naïve" $E[X^2] - E[X]^2$ qui peut donner une valeur **négative**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cancellation import (
    EPS_MACH,
    condition_addition,
    f_naive, f_stable,
    g_naive, g_stable,
    derivee_progressive,
    variance_naive, variance_deux_passes,
    tracer_beispiel_1_13, tracer_derivee_numerique, tracer_variance,
)

print(f'Précision machine ε_mach = {EPS_MACH:.3e}')

## 1. Condition de l'addition

Pour $f(x_1, x_2) = x_1 + x_2$, le script donne (formule 1.2) :
$$\text{cond}_{rel,\infty} = \frac{|x_1| + |x_2|}{|x_1 + x_2|}$$

- Si $x_1, x_2$ de même signe : numérateur = dénominateur, donc cond = 1 ✓
- Si $x_2 \approx -x_1$ : dénominateur tend vers 0, cond explose ✗

In [ ]:
for nom, a, b in [
    ('même signe',         1.0,  2.0),
    ('opposés différents', 1.0, -0.5),
    ('Auslöschung légère', 1.0, -0.99),
    ('Auslöschung forte',  1.0, -0.999_999),
    ('Auslöschung extrême',1.0, -0.999_999_999_999),
]:
    print(f'{nom:25s}  cond = {condition_addition(a, b):.3e}')

## 2. Beispiel 1.13 : $f(x) = \sqrt{x^2 + 1} - x$

Le script montre que ce problème est **bien conditionné** (condition relative ≤ 1) mais que l'algorithme naïf, qui suit l'ordre des opérations de la formule, est **instable** car la dernière étape $\sqrt{x^2+1} - x$ est une Auslöschung pour $x \gg 1$.

**Reformulation stable** (multiplication par le conjugué) :
$$\sqrt{x^2 + 1} - x \;=\; \frac{(\sqrt{x^2+1} - x)(\sqrt{x^2+1} + x)}{\sqrt{x^2+1} + x} \;=\; \frac{1}{\sqrt{x^2+1} + x}$$

Plus aucune soustraction !

In [ ]:
print(f'{"x":>10} | {"naïf":>22} | {"stable":>22} | {"err. rel. naïf":>14}')
print('-' * 78)
for k in range(1, 9):
    x = 10.0**k
    a = f_naive(x); b = f_stable(x)
    err = abs(a - b) / abs(b) if b != 0 else float('nan')
    print(f'{x:>10.0e} | {a:>22.15e} | {b:>22.15e} | {err:>14.3e}')

**Lecture du tableau :**
- À $x = 10^7$, l'algorithme naïf a déjà **0,6 % d'erreur**
- À $x = 10^8$, il renvoie **exactement 0** (alors que la vraie valeur est $5 \cdot 10^{-9}$). Erreur relative : **100 %**
- La formule stable, elle, garde toute sa précision.

In [ ]:
tracer_beispiel_1_13()
plt.show()

## 3. Un autre exemple : $(1 - \cos x) / x^2$ près de 0

Limite mathématique en 0 : $\frac{1}{2}$ (par développement de Taylor : $\cos x = 1 - x^2/2 + O(x^4)$).

**Algorithme naïf :** $\frac{1 - \cos x}{x^2}$ — Auslöschung dans $1 - \cos x$ quand $x \to 0$.

**Reformulation stable** via $1 - \cos x = 2 \sin^2(x/2)$ :
$$\frac{1 - \cos x}{x^2} = \frac{2 \sin^2(x/2)}{x^2} = \frac{1}{2} \left( \frac{\sin(x/2)}{x/2} \right)^2$$

In [ ]:
print(f'{"x":>12} | {"naïf":>20} | {"stable":>20} | {"vraie limite = 0.5":>18}')
print('-' * 78)
for k in range(1, 11):
    x = 10.0**(-k)
    print(f'{x:>12.0e} | {g_naive(x):>20.15f} | {g_stable(x):>20.15f} | {0.5:>18.15f}')

On voit que la formule naïve commence à perdre des chiffres dès $x = 10^{-4}$, et donne **0** à partir de $x = 10^{-8}$. La formule stable reste à 0,5 à toutes les échelles.

## 4. Dérivée numérique — la fameuse forme en V

Approximation : $f'(x) \approx \frac{f(x+h) - f(x)}{h}$.

Deux sources d'erreur opposées :
- **Erreur de troncature** : $O(h)$ (Taylor). Diminue quand $h$ diminue.
- **Erreur d'arrondi/Auslöschung** : $O(\varepsilon_{mach}/h)$. **Augmente** quand $h$ diminue.

L'erreur totale a un minimum atteint vers $h_{opt} \approx \sqrt{\varepsilon_{mach}} \approx 1{,}5 \cdot 10^{-8}$.

In [ ]:
tracer_derivee_numerique()
plt.show()

**Conséquence pratique :** si tu codes une dérivée numérique, ne descends jamais en dessous de $h \approx 10^{-8}$ avec des `float64`. Une intuition naïve serait de prendre $h$ aussi petit que possible — et c'est faux.

## 5. Variance : quand naïf donne **négatif**

$$\text{Var}(X) = E[X^2] - E[X]^2$$

Mathématiquement correct, numériquement catastrophique quand les deux termes sont proches (données très décalées).

**Alternative à deux passes :** $\text{Var}(X) = \frac{1}{n} \sum_i (x_i - \bar{x})^2$. On centre d'abord, donc plus de soustraction de grandes valeurs.

In [ ]:
rng = np.random.default_rng(0)
base = rng.standard_normal(10_000)  # variance ≈ 1

print(f'{"décalage":>12} | {"naïf":>14} | {"deux passes":>14}')
print('-' * 50)
for d in [0, 1, 1e3, 1e6, 1e8, 1e9, 1e10]:
    x = base + d
    print(f'{d:>12.0e} | {variance_naive(x):>14.6f} | {variance_deux_passes(x):>14.6f}')

In [ ]:
tracer_variance()
plt.show()

Pour un décalage de $10^9$, la variance naïve renvoie **−128** : une valeur négative pour ce qui est mathématiquement une somme de carrés. C'est l'Auslöschung en action.

C'est pour cette raison que `numpy.var()` n'utilise **pas** la formule naïve mais un algorithme à deux passes (et même mieux : l'algorithme de Welford pour les flux de données).

## Récapitulatif

| Situation | Symptôme | Solution |
|---|---|---|
| $\sqrt{x^2+1} - x$ pour $x$ grand | renvoie 0 | multiplier par le conjugué |
| $(1-\cos x)/x^2$ près de 0 | renvoie 0 | $2\sin^2(x/2)/x^2$ |
| dérivée avec $h$ trop petit | erreur ↑ | choisir $h \approx \sqrt{\varepsilon_{mach}}$ |
| variance $E[X^2] - E[X]^2$ | négative ! | algo à deux passes |

**Règle d'or (section 1.4.4 du script) :** *« Beim Zerlegen des Problems in Teilprobleme sollten, wenn möglich, keine schlecht konditionierten Teilprobleme verwendet werden — als Paradebeispiel die Subtraktion fast gleicher Zahlen. »*

**Prochain programme :** `gauss_elimination.py` — élimination de Gauss avec stratégies de pivot.